[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# La gramática de gráficos con *plotnine*.

## Una gramática para describir gráficos.

En 1999, el estadístico [*Leland Wilkinson*](https://en.wikipedia.org/wiki/Leland_Wilkinson) publicó *The Grammar of Graphics*, donde propuso que cualquier gráfico estadístico puede describirse como la combinación de un pequeño conjunto de componentes independientes: los datos, la forma en que las variables se asignan a propiedades visuales, los objetos geométricos que las representan, las transformaciones estadísticas, las escalas y el sistema de coordenadas.

En 2005, [*Hadley Wickham*](https://hadley.nz/) implementó estas ideas en [*ggplot2*](https://ggplot2.tidyverse.org/) para *R*, y en 2010 las refinó en el artículo [*A Layered Grammar of Graphics*](https://vita.had.co.nz/papers/layered-grammar.html): un gráfico se construye **sumando capas**. *ggplot2* se convirtió en la herramienta de visualización más usada en *R* y en un estándar de facto para la visualización estadística.

[*plotnine*](https://plotnine.org/) es la implementación de esta gramática en *Python*, con una sintaxis casi idéntica a la de *ggplot2*.

### ¿Por qué es importante?

* **Se piensa en variables, no en tipos de gráfico.** En lugar de elegir entre "gráfico de barras" o "gráfico de dispersión", se decide qué variable va en cada eje, cuál define el color y cómo se representa cada observación. Un tipo de gráfico nuevo es sólo una combinación nueva de componentes.
* **Es declarativa.** Se describe *qué* se quiere mostrar; la biblioteca decide *cómo* dibujarlo: calcula escalas, leyendas y ejes. Con *Matplotlib* (*notebooks* 28 a 31) se indica cada paso del dibujo.
* **Es un lenguaje compartido.** Las mismas ideas están en *ggplot2* (*R*), *plotnine* (*Python*), la interfaz de objetos de *Seaborn* (*notebook* 33), [*Vega-Lite*](https://vega.github.io/vega-lite/) y *Altair*. Aprender la gramática sirve en todas ellas, y mucho del material sobre visualización estadística está escrito en términos de *ggplot2*.
* **Facilita la exploración.** Cambiar un gráfico (agregar una variable como color, separar en paneles, cambiar la escala) suele ser agregar o modificar una sola capa.

[Documentación de *plotnine*](https://plotnine.org/) · [Libro *ggplot2: Elegant Graphics for Data Analysis*](https://ggplot2-book.org/)

## Los componentes de la gramática.

| Componente | En *plotnine* | Pregunta que responde |
| :--- | :--- | :--- |
| **Datos** | `ggplot(data=...)` | ¿Qué datos se grafican? |
| **Mapeos estéticos** | `aes(...)` | ¿Qué variable se asigna a cada propiedad visual (posición, color, tamaño, forma)? |
| **Geometrías** | `geom_*()` | ¿Con qué objeto se representa cada observación (puntos, líneas, barras)? |
| **Estadísticas** | `stat_*()` o `stat=` | ¿Qué cálculo se aplica a los datos antes de dibujarlos (conteo, histograma, promedio, ajuste)? |
| **Posiciones** | `position=` | ¿Cómo se acomodan los objetos que se encimarían (apilar, poner lado a lado, dispersar)? |
| **Escalas** | `scale_*()` | ¿Cómo se traducen los valores en posiciones, colores o tamaños, y cómo se rotulan? |
| **Coordenadas** | `coord_*()` | ¿En qué sistema de coordenadas se dibuja? |
| **Facetas** | `facet_wrap()`, `facet_grid()` | ¿Se divide el gráfico en paneles según una variable? |
| **Temas** | `theme_*()`, `theme()` | ¿Cómo se ven los elementos que no son datos (fondo, tipografía, rejilla)? |

Un gráfico se construye sumando componentes con `+`:

```
(
    ggplot(<datos>, aes(<mapeos>))
    + geom_<tipo>(<argumentos>)
    + scale_<estética>_<tipo>(<argumentos>)
    + facet_<tipo>(<argumentos>)
    + theme_<tipo>()
)
```

Sólo los datos, los mapeos y al menos una geometría son obligatorios; para los demás componentes hay valores por omisión razonables. Los paréntesis exteriores permiten escribir el gráfico en varias líneas.

* La siguiente celda instala *plotnine* en el entorno actual en caso de que no esté disponible.

In [ ]:
%pip install plotnine

* La siguiente celda importa los componentes de *plotnine* que se usan en este *notebook*. En la documentación de *plotnine* también es común `from plotnine import *`, que imita el estilo de *R*.

In [ ]:
import os

import pandas as pd
import polars as pl
from plotnine import (
    aes, after_stat, coord_cartesian, coord_flip, element_text, facet_grid, facet_wrap,
    geom_bar, geom_boxplot, geom_col, geom_histogram, geom_jitter, geom_line, geom_point,
    geom_smooth, geom_text, ggplot, labs, position_dodge, scale_color_brewer,
    scale_fill_brewer, scale_x_log10, scale_y_continuous, stat_summary, theme,
    theme_bw, theme_minimal,
)
from plotnine.data import mpg

dir_tmp = 'data_tmp'
os.makedirs(dir_tmp, exist_ok=True)

## Los datos.

La gramática trabaja con datos en **formato ordenado** (*tidy data*, otro concepto de *Hadley Wickham*): cada variable es una columna y cada observación es un renglón. Es el formato largo que se estudió en el *notebook* 37: si los datos están en formato ancho (por ejemplo, una columna por año), primero se convierten con `unpivot()` o `melt()`.

*plotnine* recibe *dataframes* de *Pandas* y también de *Polars*.

Los ejemplos de la primera parte usan `mpg`, el conjunto de datos clásico de los ejemplos de *ggplot2*: el rendimiento de combustible de 234 automóviles vendidos en Estados Unidos en 1999 y 2008. La última parte aplica la gramática a los datos de ventas de los *notebooks* 34 a 37.

| Columna | Descripción |
| :--- | :--- |
| `manufacturer`, `model` | Fabricante y modelo. |
| `displ` | Cilindrada del motor, en litros. |
| `year`, `cyl` | Año del modelo y número de cilindros. |
| `drv` | Tracción: `f` delantera, `r` trasera, `4` en las cuatro ruedas. |
| `cty`, `hwy` | Rendimiento en ciudad y en carretera, en millas por galón. |
| `class` | Tipo de vehículo (`compact`, `suv`, `pickup`, ...). |

In [ ]:
mpg.head()

## Mapeos estéticos.

[`aes()`](https://plotnine.org/reference/aes.html) asigna variables (columnas) a **propiedades visuales** (estéticas):

| Estética | Propiedad |
| :--- | :--- |
| `x`, `y` | Posición. |
| `color` | Color del contorno, punto o línea. |
| `fill` | Color de relleno. |
| `size` | Tamaño. |
| `shape` | Forma del punto. |
| `alpha` | Transparencia. |
| `linetype` | Tipo de línea. |
| `group` | Agrupación de las observaciones, sin efecto visual propio (por ejemplo, qué puntos une una línea). |

```
aes(x=<columna>, y=<columna>, color=<columna>, ...)
```

Donde cada `<columna>` es el nombre de una columna como cadena de texto, o una expresión sobre las columnas, como `'hwy * 0.425'` o `'factor(cyl)'`.

Los mapeos indicados en `ggplot()` se aplican a todas las capas; los indicados en una `geom_*()` sólo a esa capa.

### Mapear y fijar.

Hay una diferencia fundamental entre **mapear** una estética y **fijarla**:

* `aes(color='class')` **mapea**: el color depende del valor de la columna `class`, y se genera una leyenda.
* `geom_point(color='steelblue')` **fija**: todos los puntos tienen el mismo color, que no depende de los datos.

**Ejemplos:**

* La siguiente celda crea un gráfico sólo con datos y mapeos, sin geometría. *plotnine* ya calcula los ejes a partir de las variables, pero no dibuja ninguna observación.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy'))

* La siguiente celda agrega la geometría de puntos: cada automóvil es un punto. Se observa que los motores más grandes rinden menos en carretera.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point()

* La siguiente celda mapea la clase de vehículo al color. *plotnine* elige los colores y crea la leyenda. Los puntos con motor grande y buen rendimiento resultan ser deportivos (`2seater`).

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy', color='class')) + geom_point()

* La siguiente celda mapea dos variables más: el número de cilindros a la forma y el rendimiento en ciudad al tamaño. `factor(cyl)` indica que los cilindros se traten como categorías y no como números continuos.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy', shape='factor(cyl)', size='cty')) + geom_point(alpha=0.6)

* La siguiente celda muestra un error común: escribir un color fijo dentro de `aes()`. *plotnine* interpreta `'steelblue'` como un valor de los datos, lo mapea a un color de la paleta por omisión (no azul) y crea una leyenda innecesaria.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy', color='"steelblue"')) + geom_point()

* La forma correcta de fijar un color es como argumento de la geometría, fuera de `aes()`.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point(color='steelblue')

* Los mapeos aceptan expresiones. La siguiente celda convierte el rendimiento en carretera de millas por galón a kilómetros por litro (1 mpg ≈ 0.425 km/l) dentro del mapeo.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy * 0.425')) + geom_point() + labs(y='Rendimiento en carretera (km/l)')

## Geometrías y capas.

Las geometrías definen el objeto que representa cada observación. Un gráfico puede tener varias, una sobre otra: cada `+ geom_*()` agrega una **capa**, y las capas posteriores se dibujan encima.

| Geometría | Estéticas requeridas | Representa |
| :--- | :--- | :--- |
| [`geom_point()`](https://plotnine.org/reference/geom_point.html) | `x`, `y` | Puntos (dispersión). |
| [`geom_jitter()`](https://plotnine.org/reference/geom_jitter.html) | `x`, `y` | Puntos con un pequeño desplazamiento aleatorio, para evitar que se encimen. |
| [`geom_line()`](https://plotnine.org/reference/geom_line.html) | `x`, `y` | Líneas que unen los puntos en orden de `x`. |
| [`geom_col()`](https://plotnine.org/reference/geom_col.html) | `x`, `y` | Barras con la altura indicada por `y`. |
| [`geom_bar()`](https://plotnine.org/reference/geom_bar.html) | `x` | Barras con el **conteo** de observaciones de cada valor de `x`. |
| [`geom_histogram()`](https://plotnine.org/reference/geom_histogram.html) | `x` | Histograma de una variable continua. |
| [`geom_boxplot()`](https://plotnine.org/reference/geom_boxplot.html) | `x`, `y` | Diagrama de caja por grupo. |
| [`geom_smooth()`](https://plotnine.org/reference/geom_smooth.html) | `x`, `y` | Curva de tendencia con banda de confianza. |
| [`geom_text()`](https://plotnine.org/reference/geom_text.html) | `x`, `y`, `label` | Texto. |

[Lista completa de geometrías](https://plotnine.org/reference/#geoms)

**Ejemplos:**

* La siguiente celda combina dos capas: los puntos y una recta de regresión lineal (`method='lm'`) con su banda de confianza. Ambas capas heredan los mapeos de `ggplot()`.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point() + geom_smooth(method='lm')

* Cuando el color se mapea en `ggplot()`, todas las capas lo heredan y se ajusta una recta por cada grupo. La siguiente celda ajusta una recta por tipo de tracción.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy', color='drv')) + geom_point() + geom_smooth(method='lm', se=False)

* Si el color se mapea sólo en la capa de puntos, la curva de tendencia es una sola, para todos los datos. Es la diferencia entre mapeos globales y por capa.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point(aes(color='drv')) + geom_smooth(method='lm', color='black')

* Cada capa puede usar sus propios datos. La siguiente celda etiqueta, en una capa de texto, sólo los automóviles con rendimiento en carretera mayor a 40 mpg.

In [ ]:
eficientes = mpg[mpg['hwy'] > 40]

(
    ggplot(mpg, aes(x='displ', y='hwy'))
    + geom_point(color='gray')
    + geom_point(data=eficientes, color='darkgreen', size=3)
    + geom_text(aes(label='model'), data=eficientes, nudge_x=0.45, size=9)
)

* La siguiente celda muestra la distribución del rendimiento en carretera por clase con diagramas de caja, y encima los puntos individuales con `geom_jitter()`.

In [ ]:
(
    ggplot(mpg, aes(x='class', y='hwy'))
    + geom_boxplot(outlier_shape='')
    + geom_jitter(width=0.2, alpha=0.4)
)

## Transformaciones estadísticas.

Algunas geometrías no dibujan los datos tal cual, sino el resultado de un cálculo. Cada geometría tiene una **estadística** asociada:

| Geometría | Estadística por omisión | Cálculo |
| :--- | :--- | :--- |
| `geom_point()`, `geom_line()`, `geom_col()` | `stat_identity` | Ninguno: usa los valores tal cual. |
| `geom_bar()` | `stat_count` | Cuenta las observaciones de cada valor de `x`. |
| `geom_histogram()` | `stat_bin` | Divide `x` en intervalos y cuenta las observaciones de cada uno. |
| `geom_boxplot()` | `stat_boxplot` | Cuartiles y valores atípicos. |
| `geom_smooth()` | `stat_smooth` | Ajusta un modelo (regresión lineal, *LOESS*, ...). |

Las estadísticas calculan variables nuevas, como `count` (conteo) o `prop` (proporción). Con [`after_stat()`](https://plotnine.org/reference/after_stat.html) se pueden mapear esas variables calculadas. También es posible usar una estadística directamente con `stat_*()`, por ejemplo [`stat_summary()`](https://plotnine.org/reference/stat_summary.html) para resumir `y` por cada valor de `x`.

**Ejemplos:**

* La siguiente celda muestra la diferencia entre `geom_bar()` y `geom_col()`. `geom_bar()` cuenta los automóviles de cada clase: no necesita `y`.

In [ ]:
ggplot(mpg, aes(x='class')) + geom_bar()

* Con `geom_col()`, la altura debe estar ya calculada. La siguiente celda calcula el conteo con *Pandas* y produce el mismo gráfico.

In [ ]:
conteo = mpg['class'].value_counts().rename_axis('class').reset_index(name='n')
ggplot(conteo, aes(x='class', y='n')) + geom_col()

* La siguiente celda grafica proporciones en lugar de conteos con `after_stat('prop')`. `group=1` indica que la proporción se calcule respecto del total y no dentro de cada clase.

In [ ]:
ggplot(mpg, aes(x='class', y=after_stat('prop'), group=1)) + geom_bar()

* La siguiente celda dibuja un histograma con intervalos de 2 mpg.

In [ ]:
ggplot(mpg, aes(x='hwy')) + geom_histogram(binwidth=2, fill='steelblue', color='white')

* La siguiente celda usa `stat_summary()` para mostrar el rendimiento promedio por clase con su intervalo de confianza del 95 %, en lugar de los puntos individuales.

In [ ]:
ggplot(mpg, aes(x='class', y='hwy')) + stat_summary(fun_data='mean_cl_normal')

## Posiciones.

Cuando varias barras o puntos ocupan el mismo lugar, el argumento `position` de la geometría define cómo se acomodan:

| Posición | Efecto |
| :--- | :--- |
| `'stack'` | Apila (por omisión en barras). |
| `'dodge'` o `position_dodge()` | Coloca lado a lado. |
| `'fill'` | Apila y normaliza la altura total a 1, para comparar proporciones. |
| `'jitter'` | Desplaza los puntos aleatoriamente (equivale a `geom_jitter()`). |
| `'identity'` | Deja cada objeto en su lugar, aunque se encimen. |

**Ejemplos:**

* La siguiente celda cuenta los automóviles por clase y tipo de tracción con barras apiladas (posición por omisión).

In [ ]:
ggplot(mpg, aes(x='class', fill='drv')) + geom_bar()

* La siguiente celda coloca las barras lado a lado.

In [ ]:
ggplot(mpg, aes(x='class', fill='drv')) + geom_bar(position=position_dodge(preserve='single'))

* La siguiente celda normaliza cada barra a 1 para comparar la proporción de cada tipo de tracción dentro de cada clase.

In [ ]:
ggplot(mpg, aes(x='class', fill='drv')) + geom_bar(position='fill') + labs(y='proporción')

## Escalas.

Cada estética mapeada tiene una **escala**, que traduce los valores de los datos a posiciones, colores o tamaños, y genera los ejes y leyendas. *plotnine* elige una escala por omisión según el tipo de variable; las funciones `scale_<estética>_<tipo>()` permiten cambiarla.

| Escala | Uso |
| :--- | :--- |
| `scale_x_continuous()`, `scale_y_continuous()` | Ejes numéricos: límites (`limits`), marcas (`breaks`) y rótulos (`labels`). |
| `scale_x_log10()`, `scale_y_log10()` | Ejes en escala logarítmica. |
| `scale_x_discrete()`, `scale_y_discrete()` | Ejes categóricos: orden y rótulos. |
| `scale_color_brewer()`, `scale_fill_brewer()` | Paletas de [*ColorBrewer*](https://colorbrewer2.org/), diseñadas para mapas y gráficos. |
| `scale_color_manual()`, `scale_fill_manual()` | Colores elegidos a mano. |
| `scale_color_gradient()`, `scale_fill_gradient()` | Degradados para variables continuas. |

[`labs()`](https://plotnine.org/reference/labs.html) cambia los títulos del gráfico, de los ejes y de las leyendas.

**Ejemplos:**

* La siguiente celda usa una paleta cualitativa de *ColorBrewer*, rotula el eje $y$ con unidades y agrega títulos.

In [ ]:
(
    ggplot(mpg, aes(x='displ', y='hwy', color='drv'))
    + geom_point()
    + scale_color_brewer(type='qual', palette='Set1', labels=['4x4', 'delantera', 'trasera'])
    + scale_y_continuous(labels=lambda valores: [f'{v:.0f} mpg' for v in valores])
    + labs(
        title='Rendimiento en carretera según la cilindrada',
        x='Cilindrada (litros)',
        y='Rendimiento en carretera',
        color='Tracción',
    )
)

* La siguiente celda usa una escala logarítmica en el eje $x$.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point() + scale_x_log10()

## Sistemas de coordenadas.

| Función | Efecto |
| :--- | :--- |
| `coord_cartesian(xlim=..., ylim=...)` | Coordenadas cartesianas (por omisión). Con límites, hace un **acercamiento** sin descartar datos. |
| `coord_flip()` | Intercambia los ejes; útil para barras horizontales con etiquetas largas. |

Hay una diferencia importante entre limitar la **escala** y limitar las **coordenadas**:

* `scale_y_continuous(limits=(20, 30))` **elimina** los datos fuera del rango antes de calcular las estadísticas. Una recta de regresión, por ejemplo, se ajusta sólo con los datos restantes.
* `coord_cartesian(ylim=(20, 30))` calcula todo con los datos completos y sólo **acerca la vista**.

**Ejemplos:**

* La siguiente celda limita la escala: la recta se ajusta sólo con los puntos entre 20 y 30 mpg, y *plotnine* advierte que eliminó datos.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point() + geom_smooth(method='lm') + scale_y_continuous(limits=(20, 30))

* La siguiente celda limita las coordenadas: la recta es la que se ajusta con todos los datos, y sólo cambia la vista.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point() + geom_smooth(method='lm') + coord_cartesian(ylim=(20, 30))

* La siguiente celda dibuja barras horizontales con `coord_flip()`.

In [ ]:
ggplot(mpg, aes(x='manufacturer')) + geom_bar() + coord_flip()

## Facetas.

Las facetas dividen un gráfico en **paneles**, uno por cada valor de una o dos variables, con los mismos ejes. Permiten comparar grupos sin encimar sus datos (lo que en inglés se llama *small multiples*).

| Función | Paneles |
| :--- | :--- |
| [`facet_wrap('<variable>', ncol=..., scales=...)`](https://plotnine.org/reference/facet_wrap.html) | Uno por cada valor de una variable, acomodados en filas. |
| [`facet_grid('<fila> ~ <columna>', scales=...)`](https://plotnine.org/reference/facet_grid.html) | Una cuadrícula: filas según una variable y columnas según otra. |

`scales='free'`, `'free_x'` o `'free_y'` permite que cada panel tenga su propia escala.

**Ejemplos:**

* La siguiente celda crea un panel por clase de vehículo.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point() + facet_wrap('class', ncol=4)

* La siguiente celda crea una cuadrícula: tipo de tracción en las filas y año del modelo en las columnas.

In [ ]:
ggplot(mpg, aes(x='displ', y='hwy')) + geom_point() + geom_smooth(method='lm', se=False) + facet_grid('drv ~ year')

## Temas.

Los temas controlan los elementos que no dependen de los datos: fondo, rejilla, tipografía, posición de la leyenda, etc.

| Función | Apariencia |
| :--- | :--- |
| `theme_gray()` | Fondo gris con rejilla blanca (por omisión, como en *ggplot2*). |
| `theme_bw()` | Fondo blanco con rejilla gris y marco. |
| `theme_minimal()` | Sin marco ni fondo; rejilla tenue. |
| `theme_classic()` | Sin rejilla, con líneas de ejes. |
| `theme_xkcd()`, `theme_tufte()`, ... | Otros estilos. |

[`theme()`](https://plotnine.org/reference/theme.html) ajusta elementos individuales. Sus argumentos usan guiones bajos donde *ggplot2* usa puntos: `axis_text_x` en *plotnine* equivale a `axis.text.x` en *R*. `theme_set()` fija un tema para todos los gráficos siguientes.

**Ejemplo:**

* La siguiente celda aplica `theme_minimal()`, mueve la leyenda abajo y rota las etiquetas del eje $x$.

In [ ]:
(
    ggplot(mpg, aes(x='class', fill='drv'))
    + geom_bar()
    + theme_minimal()
    + theme(legend_position='bottom', axis_text_x=element_text(rotation=45, ha='right'))
)

## Los gráficos son objetos.

Un gráfico de *plotnine* es un objeto de *Python* (de clase `ggplot`): se puede guardar en una variable, agregarle capas después y reutilizarlo como base de varios gráficos. Sólo se dibuja cuando se muestra o se guarda.

[`<gráfico>.save(<ruta>, width=..., height=..., dpi=...)`](https://plotnine.org/reference/ggplot.html#plotnine.ggplot.save) guarda el gráfico en un archivo; el formato se deduce de la extensión (`.png`, `.pdf`, `.svg`, ...). `width` y `height` están en pulgadas.

* La siguiente celda define un gráfico base y crea dos variantes a partir de él.

In [ ]:
base = ggplot(mpg, aes(x='displ', y='hwy')) + geom_point(alpha=0.5) + theme_bw()

con_tendencia = base + geom_smooth(method='lm')
por_traccion = base + facet_wrap('drv')
con_tendencia

In [ ]:
por_traccion

* La siguiente celda guarda la variante con tendencia en un archivo *PNG*.

In [ ]:
ruta = os.path.join(dir_tmp, 'rendimiento.png')
con_tendencia.save(ruta, width=6, height=4, dpi=150, verbose=False)
os.path.exists(ruta)

## Caso práctico: los datos de ventas.

Esta sección aplica la gramática a los datos de ventas de los *notebooks* 34 a 37. Los datos se preparan con *Polars* y se grafican directamente; para cada gráfico conviene pensar primero en la pregunta y después en los componentes: qué variable va en cada eje, cuál define el color o los paneles, y qué geometría y estadística corresponden.

* La siguiente celda lee las ventas, calcula el importe neto y agrega el mes de cada venta.

In [ ]:
ventas = (
    pl.read_csv(os.path.join('data', 'ventas.csv'), try_parse_dates=True)
    .join(pl.read_csv(os.path.join('data', 'tiendas.csv')).select('tienda_id', 'region'), on='tienda_id', how='inner')
    .with_columns(
        importe_neto=pl.col('cantidad') * pl.col('precio_unitario') * (1 - pl.col('descuento').fill_null(0)),
        mes=pl.col('fecha_hora').dt.truncate('1mo').dt.date(),
    )
)
ventas.select('mes', 'region', 'canal', 'categoria', 'importe_neto').head(3)

* **¿Cómo evolucionan las ventas de cada categoría?** Tiempo en $x$, importe en $y$, una línea por categoría (color) y un panel por categoría (facetas) para que las escalas distintas no oculten las tendencias.

In [ ]:
mensual = ventas.group_by('mes', 'categoria').agg((pl.col('importe_neto').sum() / 1e6).alias('millones'))

(
    ggplot(mensual, aes(x='mes', y='millones', color='categoria'))
    + geom_line()
    + geom_smooth(method='lm', se=False, linetype='dashed', size=0.5)
    + facet_wrap('categoria', scales='free_y')
    + labs(title='Importe mensual por categoría', x='', y='Millones de pesos')
    + theme_bw()
    + theme(legend_position='none', axis_text_x=element_text(rotation=45, ha='right'))
)

* **¿Qué región vende más y por qué canal?** Región en $x$, importe en $y$ y canal en el relleno, con barras lado a lado.

In [ ]:
por_region = ventas.group_by('region', 'canal').agg((pl.col('importe_neto').sum() / 1e6).alias('millones'))

(
    ggplot(por_region, aes(x='region', y='millones', fill='canal'))
    + geom_col(position='dodge')
    + scale_fill_brewer(type='qual', palette='Set2')
    + labs(title='Importe 2024-2025 por región y canal', x='Región', y='Millones de pesos', fill='Canal')
    + theme_minimal()
)

* **¿Cómo se distribuye el importe de una venta en cada categoría?** Categoría en $x$, importe en $y$, diagramas de caja y escala logarítmica, porque los importes van de decenas a miles de pesos.

In [ ]:
(
    ggplot(ventas, aes(x='categoria', y='importe_neto'))
    + geom_boxplot()
    + scale_y_continuous(trans='log10')
    + coord_flip()
    + labs(title='Distribución del importe por venta', x='', y='Importe neto (escala logarítmica)')
    + theme_bw()
)

## De *ggplot2* a *plotnine*.

Como la sintaxis es casi la misma, los ejemplos de *ggplot2* (libros, cursos, respuestas en foros) se pueden traducir a *plotnine* con pocos cambios:

| *ggplot2* (*R*) | *plotnine* (*Python*) |
| :--- | :--- |
| `ggplot(mpg, aes(x = displ, y = hwy))` | `ggplot(mpg, aes(x='displ', y='hwy'))`: los nombres de columna van entre comillas. |
| Un gráfico en varias líneas termina cada línea con `+` | Se encierra el gráfico entre paréntesis, o cada línea termina con `+` dentro de ellos. |
| `facet_wrap(~ class)` | `facet_wrap('class')` o `facet_wrap('~ class')` |
| `theme(axis.text.x = element_text(angle = 45))` | `theme(axis_text_x=element_text(rotation=45))`: guiones bajos en lugar de puntos. |
| `geom_point(colour = "red")` | `geom_point(color='red')` (también acepta `colour`). |
| `ggsave("grafico.png", p)` | `p.save('grafico.png')` |
| `after_stat(prop)` | `after_stat('prop')` |

[Guía de *plotnine* para usuarios de *ggplot2*](https://plotnine.org/guide/introduction.html)

## La gramática en *Seaborn*.

La interfaz `seaborn.objects` (*notebook* 33) aplica la misma gramática con otra sintaxis: en lugar de sumar componentes con `+`, se encadenan métodos de un objeto `so.Plot`.

| Componente | *plotnine* | `seaborn.objects` |
| :--- | :--- | :--- |
| Datos y mapeos | `ggplot(df, aes(x=..., y=..., color=...))` | `so.Plot(df, x=..., y=..., color=...)` |
| Geometría | `+ geom_point()` | `.add(so.Dot())` |
| Estadística | `+ geom_histogram()` | `.add(so.Bars(), so.Hist())` |
| Agregación con intervalo | `+ stat_summary(fun_data='mean_cl_normal')` | `.add(so.Range(), so.Est())` y `.add(so.Dot(), so.Agg())` |
| Tendencia | `+ geom_smooth(method='lm')` | `.add(so.Line(), so.PolyFit(order=1))` |
| Posición | `position='dodge'`, `'stack'`, `'jitter'` | `so.Dodge()`, `so.Stack()`, `so.Jitter()` |
| Escalas | `+ scale_x_log10()`, `+ scale_color_brewer(...)` | `.scale(x='log', color='Set2')` |
| Facetas | `+ facet_wrap(...)`, `+ facet_grid(...)` | `.facet(col=..., row=...)` |
| Límites de la vista | `+ coord_cartesian(xlim=...)` | `.limit(x=...)` |
| Otras coordenadas | `coord_flip()`, ... | No disponibles |
| Títulos | `+ labs(...)` | `.label(...)` |
| Apariencia | `+ theme_*()`, `+ theme(...)` | `.theme(...)` |
| Guardado | `.save(...)` | `.save(...)` |

`seaborn.objects` es más reciente y aún se considera experimental; *plotnine* cubre la gramática de forma más completa y es la referencia más cercana a *ggplot2*.

## Resumen.

| Componente | En *plotnine* | Idea clave |
| :--- | :--- | :--- |
| **Datos** | `ggplot(data)` | Formato ordenado: una variable por columna, una observación por renglón. |
| **Mapeos** | `aes()` | Asignan variables a propiedades visuales. Mapear (dentro de `aes()`) no es lo mismo que fijar (fuera). |
| **Geometrías** | `geom_*()` | Cada una es una capa; las capas heredan los mapeos globales. |
| **Estadísticas** | `stat_*()`, `after_stat()` | Cálculos previos al dibujo: conteos, intervalos, resúmenes, ajustes. |
| **Posiciones** | `position=` | Apilar, poner lado a lado, normalizar o dispersar. |
| **Escalas** | `scale_*()`, `labs()` | Traducción de valores a ejes y colores; rótulos. |
| **Coordenadas** | `coord_*()` | Limitar coordenadas acerca la vista; limitar escalas elimina datos. |
| **Facetas** | `facet_wrap()`, `facet_grid()` | Un panel por grupo para comparar. |
| **Temas** | `theme_*()`, `theme()` | Apariencia de lo que no son datos. |

La gramática de gráficos es una forma de **pensar** las visualizaciones que se aplica más allá de *plotnine*: en *ggplot2*, en la interfaz de objetos de *Seaborn* y en *Vega-Lite*/*Altair*.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>